# 08 — Multi-city Standardization

Chuẩn hóa 5 city từ UCI Five Chinese Cities. Raw files nằm trực tiếp dưới Project/data/raw/.

In [ ]:
from pathlib import Path
import pandas as pd,numpy as np
PROJECT_ROOT=Path.cwd().parents[2]; RAW_DIR=PROJECT_ROOT/'data'/'raw'; SON_OUTPUT=PROJECT_ROOT/'notebooks'/'son_nb'/'output'; WORK_DATA=SON_OUTPUT/'data'
FILES={'beijing':'BeijingPM20100101_20151231.csv','chengdu':'ChengduPM20100101_20151231.csv','guangzhou':'GuangzhouPM20100101_20151231.csv','shanghai':'ShanghaiPM20100101_20151231.csv','shenyang':'ShenyangPM20100101_20151231.csv'}
for city,f in FILES.items(): print(city, RAW_DIR/f, (RAW_DIR/f).exists())

### Data Contract v1
`city_id, country, datetime, pm25, DEWP, HUMI, TEMP, PRES, cbwd, Iws, precipitation, Iprec`

Meta-features city sẽ để bảng riêng và thêm sau khi có nguồn đáng tin cậy. Không tự bịa.

In [ ]:
def build_datetime(df): return pd.to_datetime(df[['year','month','day','hour']])
def first_existing(df,cols):
    for c in cols:
        if c in df.columns: return c
    return None
def standardize(path,city):
    raw=pd.read_csv(path); raw['datetime']=build_datetime(raw); pm=first_existing(raw,['PM_US Post','PM_Dongsi','PM_Dongsihuan','PM_Nongzhanguan'])
    if pm is None: raise ValueError(f'No PM column: {path.name}')
    return pd.DataFrame({'city_id':city,'country':'China','datetime':raw.datetime,'pm25':raw[pm],'DEWP':raw.DEWP if 'DEWP' in raw else np.nan,'HUMI':raw.HUMI if 'HUMI' in raw else np.nan,'TEMP':raw.TEMP if 'TEMP' in raw else np.nan,'PRES':raw.PRES if 'PRES' in raw else np.nan,'cbwd':raw.cbwd if 'cbwd' in raw else np.nan,'Iws':raw.Iws if 'Iws' in raw else np.nan,'precipitation':raw.precipitation if 'precipitation' in raw else np.nan,'Iprec':raw.Iprec if 'Iprec' in raw else np.nan,'pm_column_used':pm,'source_file':path.name}).sort_values('datetime').reset_index(drop=True)
frames=[]
for city,f in FILES.items():
    p=RAW_DIR/f
    if p.exists(): frames.append(standardize(p,city))
if frames:
    multi=pd.concat(frames,ignore_index=True); print('shape=',multi.shape); display(multi.head()); multi.to_csv(WORK_DATA/'five_cities_standardized_working.csv',index=False)
else: print('No Five-City files found.')

### Quan trọng
Không tự động thay baseline Beijing standalone bằng Beijing của Five-City. Trước tiên kiểm tra `standalone pm2.5` và `PM_US Post` trong giai đoạn overlap 2010–2014.